## Tahap 0.1 — Lingkungan

**Dasar naskah:** Bab III (lingkungan komputasi: Google Colab) dan Bab III tahap 10 (GPU **T4**). `REVISI_30OKT.md` R1 (data CSV), R2 (folder kerja `MyDrive/skripsi_wondr_v4/`; `skripsi_wondr_v3/` tidak disentuh), R11 (seluruh pelatihan di T4). `CLAUDE.md` Bagian 3.1, 3.3, 3.4.

**Package:** `os`, `sys`, `subprocess`, `importlib`, `warnings`, `pathlib`, `json`, `time`, `platform` (bawaan Python); `numpy`, `pandas`, `torch`, `transformers`, `matplotlib` (di-*import*); `scikit-learn`, `scipy`, `openpyxl`, `pyarrow` (dipasang bila belum ada, versinya dicatat); `google.colab.drive` (*mount* Google Drive).

Jalankan sel 1/2 lalu sel 2/2 **paling awal** setiap kali *runtime* dimulai atau di-*restart*, sebelum Tahap 0.2.

In [ ]:
# ══ Tahap 0.1 — Lingkungan (sel 1/2: variabel lingkungan, folder kerja, pemasangan package) ══
# Package: os, sys, subprocess, importlib, warnings, pathlib (bawaan Python); google.colab.drive
import os, sys, subprocess, importlib.util, warnings
from pathlib import Path
from google.colab import drive

# 1) Variabel lingkungan, diatur SEBELUM torch/transformers di-import (3.3–3.4)
_cuda_dulu = "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized()
_cublas_dulu = os.environ.get("CUBLAS_WORKSPACE_CONFIG")
os.environ.update({
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",                                          # determinisme cuBLAS
    "TOKENIZERS_PARALLELISM": "false",                                             # senyap: tokenizer
    "TRANSFORMERS_VERBOSITY": "error", "TRANSFORMERS_NO_ADVISORY_WARNINGS": "1",   # senyap: transformers
    "HF_HUB_DISABLE_PROGRESS_BARS": "1", "HF_DATASETS_DISABLE_PROGRESS_BARS": "1", # senyap: HF Hub, datasets
    "TQDM_DISABLE": "1",                                                           # senyap: semua tqdm
    "USE_TF": "0", "TF_CPP_MIN_LOG_LEVEL": "3",                                    # jangan muat TensorFlow
})
warnings.filterwarnings("ignore")

# 2) Google Drive dan struktur folder (3.1); folder versi lama tidak disentuh
if not os.path.isdir("/content/drive/MyDrive"):
    try:
        drive.mount("/content/drive")
    except ValueError as e:
        raise RuntimeError("Mount gagal: pilih akun yang menyimpan folder skripsi, centang SEMUA izin, ulangi sel.") from e
AKAR = Path("/content/drive/MyDrive/skripsi_wondr_v4")    # R2: folder v4; skripsi_wondr_v3 tidak disentuh
AKAR.mkdir(exist_ok=True)                         # gagal jika MyDrive belum ter-mount
for sub in ["data", "lib", "hasil", "anotasi", "bab4/tabel", "bab4/gambar", "log"]:
    (AKAR / sub).mkdir(parents=True, exist_ok=True)
(AKAR / "lib" / "__init__.py").touch()            # agar "from lib.wondr_teks import ..." jalan
os.chdir(AKAR)                                     # agar %%writefile cukup memakai lib/...
if str(AKAR) not in sys.path:
    sys.path.insert(0, str(AKAR))

# 3) Pasang hanya package yang belum ada (di-import di sel 2/2)
PAKET = {"numpy": "numpy", "pandas": "pandas", "torch": "torch", "transformers": "transformers",
         "sklearn": "scikit-learn", "scipy": "scipy", "matplotlib": "matplotlib",
         "openpyxl": "openpyxl", "pyarrow": "pyarrow"}            # nama modul -> nama pip
_dipasang = [pip for mod, pip in PAKET.items() if importlib.util.find_spec(mod) is None]
if _dipasang:
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_dipasang], capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError("pip install gagal:\n" + _r.stderr[-2000:])

In [ ]:
# ══ Tahap 0.1 — Lingkungan (sel 2/2: import, GPU, dan log versi) ══
# Package: importlib, json, time, platform (bawaan Python); numpy, pandas, torch, transformers,
#          matplotlib (di-import); scikit-learn, scipy, openpyxl, pyarrow (versinya dicatat)
import importlib.metadata, json, time, platform
t0 = time.perf_counter()
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, transformers
from huggingface_hub.utils import logging as hf_logging

# 1) Matikan kebisingan yang tersisa setelah import
transformers.logging.set_verbosity_error()        # sembunyikan "Some weights ... newly initialized"
transformers.logging.disable_progress_bar()       # progress bar unduhan model
hf_logging.set_verbosity_error()                  # sembunyikan peringatan HF Hub "unauthenticated requests"
plt.rcParams.update({"savefig.dpi": 300, "savefig.bbox": "tight"})   # PNG 300 dpi (3.1)

# 2) GPU dan versi -> log/versi_package.json (acuan = sesi T4 pertama)
GPU_PENELITIAN = "T4"                              # R11: Bab III tahap 10; seluruh pelatihan di T4
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada"
mem = torch.cuda.get_device_properties(0).total_memory / 1024**3 if gpu != "tidak ada" else 0.0
sesi = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(), "gpu": gpu,
        "gpu_memori_gb": round(mem, 2), "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
        "package": {p: importlib.metadata.version(p) for p in [*PAKET.values(), "tokenizers", "huggingface-hub"]}}
log = AKAR / "log" / "versi_package.json"
lama = json.loads(log.read_text()) if log.exists() else {}
pertama = lama["pertama"] if GPU_PENELITIAN in lama.get("pertama", {}).get("gpu", "") else sesi
log.write_text(json.dumps({"pertama": pertama, "terbaru": sesi}, indent=2))
beda = [k for k in ("torch", "transformers", "tokenizers") if pertama["package"][k] != sesi["package"][k]]
beda += ["GPU"] if pertama["gpu"] != gpu else []

# 3) Ringkasan + cek (judul() dan cek() baru dibuat di Tahap 0.2)
csv, koma = AKAR / "data" / "review_wondr_bni_databaruu.csv", lambda x: f"{x:.1f}".replace(".", ",")   # R1
cublas_ok = not _cuda_dulu or _cublas_dulu == ":4096:8"
print(f"══════════ TAHAP 0.1 — LINGKUNGAN ══════════\nMasukan : data/{csv.name}\nKeluaran: log/{log.name}")
print(f"{'─' * 48}\nFolder kerja : {AKAR}\nPython       : {sesi['python']}\n"
      f"GPU          : {gpu} ({koma(mem)} GB) · CUDA {sesi['cuda']} · cuDNN {sesi['cudnn']}\n\n{'Package':<17}Versi")
print("\n".join(f"{k:<17}{v}{'  (baru dipasang)' * (k in _dipasang)}" for k, v in sesi["package"].items()))
print(f"{'─' * 48}\nCek otomatis:")
for nama, ok, aktual, target in [
        ("berkas data ada", csv.exists(), "ya" if csv.exists() else "tidak", "ya"),
        ("GPU", GPU_PENELITIAN in gpu, gpu, f"memuat {GPU_PENELITIAN}"),
        ("CUBLAS_WORKSPACE_CONFIG diatur sebelum CUDA aktif", cublas_ok, "ya" if cublas_ok else "tidak, restart runtime", "ya"),
        ("versi torch/transformers/tokenizers/GPU", not beda, "berubah: " + ", ".join(beda) if beda else "sama", "sama")]:
    print(f"  {'✅' if ok else '⚠️'} {nama} = {aktual} (target {target})")
print(f"Waktu: {koma(time.perf_counter() - t0)} detik")

## Tahap 0.2 — Konfigurasi dan fungsi bantu

**Dasar naskah:** Bab III (seluruh tahap; Tabel 3.1 dan 3.2). `CLAUDE.md` Bagian 4 dan keputusan KT-2, KT-3, KT-4; `REVISI_30OKT.md` R3 (*seed*: RS 1–10, PSO 11–20, model akhir 44–48), R4 (10 *run* per metode), R10 (`kembaran_dekat` dihapus), R11 (GPU T4), Bagian 3 (target data baru), Bagian 6 (nomor persamaan).

**Package:** `importlib` (bawaan Python); `lib.bantu` (`json`, `math`, `os`, `random`, `time`, `pathlib`, `numpy`; `torch` dan `transformers` di `atur_seed()`).

Sel 1/2 menulis `lib/bantu.py` (tidak berubah dari v3). Sel 2/2 membuat `PATH`, `CFG`, dan `TARGET`; jalankan setelah Tahap 0.1 setiap kali *runtime* dimulai.

In [ ]:
%%writefile lib/bantu.py
# lib/bantu.py -- fungsi bantu untuk semua subtahap (Tahap 0.2; KT-4 disetujui 21-10-2026).
# Isinya fungsi saja; PATH, CFG, dan TARGET dibuat di sel 2/2 Tahap 0.2.
import json, math, os, random, time
from pathlib import Path
import numpy as np

__all__ = ["fmt_id", "rel", "mulai", "judul", "garis", "cek", "selesai",
           "simpan_tabel", "simpan_gambar", "simpan_json", "baca_json", "atur_seed"]
_AKAR = Path(__file__).resolve().parents[1]          # folder kerja .../skripsi_wondr_v3
_STATUS = {"t0": None, "cek_mulai": False}
_LEBAR = 48                                           # lebar garis pemisah blok ringkasan


def _angka(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def fmt_id(x, d=None):
    """Format angka Indonesia: 7204 -> '7.204'; 0.836 -> '0,8360'. Selain angka -> str(x)."""
    if not _angka(x):
        return str(x)
    if d is None:
        d = 0 if isinstance(x, (int, np.integer)) else 4
    return f"{x:,.{d}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def _tampil(v, d=None):
    """Nilai untuk ringkasan: bilangan bulat tanpa desimal, pecahan d desimal, daftar dipisah ' · '."""
    if isinstance(v, np.ndarray):
        v = v.tolist()
    if isinstance(v, dict):
        return ", ".join(f"{k}: {_tampil(x, d)}" for k, x in v.items())
    if isinstance(v, (list, tuple)):
        return " · ".join(f"[{_tampil(x, d)}]" if isinstance(x, (list, tuple)) else _tampil(x, d) for x in v)
    return fmt_id(v) if isinstance(v, (int, np.integer)) else fmt_id(v, d)


def rel(p):
    """Path relatif terhadap folder kerja, untuk ditampilkan."""
    try:
        return str(Path(p).resolve().relative_to(_AKAR))
    except ValueError:
        return str(p)


def garis():
    print("─" * _LEBAR)


def mulai():
    """Dipanggil di awal sel: mulai menghitung waktu."""
    _STATUS["t0"] = time.perf_counter()


def judul(no, nama, masukan=None, keluaran=None):
    """Kepala blok ringkasan; masukan/keluaran berupa str atau list of str."""
    _STATUS["cek_mulai"] = False
    print(f"{'═' * 10} TAHAP {no} — {nama.upper()} {'═' * 10}")
    for label, isi in (("Masukan ", masukan), ("Keluaran", keluaran)):
        if isi:
            isi = [isi] if isinstance(isi, str) else list(isi)
            print(f"{label}: " + f"\n{' ' * 10}".join(isi))
    garis()


def _sama(a, t, tol):
    a = a.tolist() if isinstance(a, np.ndarray) else a
    t = t.tolist() if isinstance(t, np.ndarray) else t
    if _angka(a) and _angka(t):
        return abs(float(a) - float(t)) <= tol
    if isinstance(a, dict) and isinstance(t, dict):
        return a.keys() == t.keys() and all(_sama(a[k], t[k], tol) for k in t)
    if isinstance(a, (list, tuple)) and isinstance(t, (list, tuple)):
        return len(a) == len(t) and all(_sama(x, y, tol) for x, y in zip(a, t))
    return bool(a == t)


def cek(nama, aktual, target, toleransi=0, cara="=", d=None):
    """Cetak ✅/⚠️ beserta nilai aktual dan target, tanpa assert. cara: '=', '>', '>=', '<', '<='."""
    if not _STATUS["cek_mulai"]:
        garis()
        print("Cek otomatis:")
        _STATUS["cek_mulai"] = True
    if d is None and toleransi > 0:
        d = max(0, round(-math.log10(2 * toleransi)))  # 0,05 -> 1 desimal; 5e-5 -> 4 desimal
    if cara == "=":
        ok = _sama(aktual, target, toleransi)
    else:
        a, t = float(aktual), float(target)
        ok = {">": a > t, ">=": a >= t, "<": a < t, "<=": a <= t}[cara]
    tanda = "" if cara == "=" else cara + " "
    print(f"  {'✅' if ok else '⚠️'} {nama} = {_tampil(aktual, d)} (target {tanda}{_tampil(target, d)})")
    return bool(ok)


def selesai():
    """Baris terakhir blok ringkasan: lama sel berjalan sejak mulai()."""
    dt = time.perf_counter() - _STATUS["t0"] if _STATUS["t0"] else 0.0
    teks = (f"{fmt_id(dt, 1)} detik" if dt < 120 else
            f"{fmt_id(dt / 60, 1)} menit" if dt < 7200 else f"{fmt_id(dt / 3600, 2)} jam")
    print(f"Waktu: {teks}")


def simpan_tabel(df, nama, index=False):
    """CSV untuk Bab IV (angka mentah) -> bab4/tabel/<nama>.csv; mengembalikan path-nya."""
    p = _AKAR / "bab4" / "tabel" / f"{nama}.csv"
    df.to_csv(p, index=index)
    return p


def simpan_gambar(fig, nama):
    """PNG 300 dpi -> bab4/gambar/<nama>.png, lalu ditampilkan satu kali."""
    import matplotlib.pyplot as plt
    p = _AKAR / "bab4" / "gambar" / f"{nama}.png"
    fig.savefig(p, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return p


def _ke_json(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, (np.ndarray, Path)):
        return o.tolist() if isinstance(o, np.ndarray) else str(o)
    raise TypeError(f"tidak bisa disimpan ke JSON: {type(o).__name__}")


def simpan_json(obj, path):
    """Tulis JSON secara atomik (berkas .tmp lalu diganti), aman bila Colab terputus."""
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_ke_json), encoding="utf-8")
    os.replace(tmp, path)
    return path


def baca_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def atur_seed(s):
    """Satu seed untuk semua sumber keacakan pelatihan (3.4), ditambah KT-2 dan KT-3 (21-10-2026)."""
    import torch, transformers
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    transformers.set_seed(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)         # KT-2: operasi GPU tak-deterministik -> error
    torch.backends.cuda.matmul.allow_tf32 = False    # KT-3: FP32 penuh; API lama ikut mengatur API baru,
    torch.backends.cudnn.allow_tf32 = False          #       API baru saja tidak mematikan TF32 konv/RNN cuDNN


def uji_bantu(s):
    """Uji Tahap 0.2: atur_seed(s) dua kali -> angka acak sama; status mode deterministik dan TF32."""
    import torch
    atur_seed(s); a = np.random.rand(3), torch.rand(3)
    atur_seed(s); b = np.random.rand(3), torch.rand(3)
    return {"ulang_sama": all(np.array_equal(x, y) for x, y in zip(a, b)),
            "deterministik": torch.are_deterministic_algorithms_enabled(),
            "tf32": [torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32]}

In [ ]:
# ══ Tahap 0.2 — Konfigurasi dan fungsi bantu (sel 2/2: PATH, CFG, TARGET) ══
# Package: importlib (bawaan Python); lib.bantu (fungsi bantu dari sel 1/2)
import importlib, lib.bantu
importlib.reload(lib.bantu)                         # pakai isi lib/bantu.py yang terbaru
from lib.bantu import *
mulai()

# 1) Lokasi folder dan berkas (3.1)
PATH = {"akar": AKAR, **{k: AKAR / k for k in ("data", "lib", "hasil", "anotasi", "log")},
        "tabel": AKAR / "bab4/tabel", "gambar": AKAR / "bab4/gambar", "csv": AKAR / "data/review_wondr_bni_databaruu.csv"}

# 2) Konfigurasi tetap (CLAUDE.md Bagian 4; REVISI_30OKT R3, R4, R11) -- jangan diubah
CFG = {
    "gpu": "T4",                                                    # R11; Bab III tahap 10
    "checkpoint": "indobenchmark/indobert-base-p1",                 # Bab III tahap 7; Wilie dkk. (2020)
    "tau": 4, "kelas": {0: "Non-Positif", 1: "Positif"},            # pers. (2.1)-(2.2): rating >= 4 -> 1
    "proporsi": {"latih": 0.70, "validasi": 0.15, "uji": 0.15},     # Bab III tahap 6
    "seed_bagi": 42, "kelompok_panjang": ("1-2", "3-10", ">10"),    # Bab III tahap 6
    "q": 100, "b": 32,                                              # pers. (2.9): L_max dari data latih saja
    "cl_lipatan": 5, "cl_seed_lipatan": 42, "cl_seed_latih": 42,    # Bab III tahap 8; Keputusan teknis
    "seed_anotasi": 2026,                                           # Keputusan: urutan lembar anotasi (tahap 9)
    "B": 16, "E": 4, "p": 2, "v": 1.0,                              # Tabel 3.1: batch, epoch, patience, norma
    "adam_betas": (0.9, 0.999), "adam_eps": 1e-8,                   # pers. (2.18); bawaan PyTorch
    "default": {"lr": 2e-5, "wr": 0.1, "wd": 0.01},                 # Tabel 3.1; Devlin dkk.; Sun dkk.
    "ruang": {"lr": (1e-5, 5e-5), "wr": (0.0, 0.2), "wd": (0.0, 0.3)},   # Tabel 3.1; pers. (2.29); lr skala log
    "seed_fitness": (42, 43),                                       # Bab III tahap 10; pers. (2.23), (2.25)-(2.26)
    "rs_M": 50, "rs_seeds": tuple(range(1, 11)),                    # Bab III tahap 11; R3-R4: 10 run, seed 1-10
    "pso": {"s": 10, "G_max": 4, "omega": 0.729, "c1": 1.49445, "c2": 1.49445, "V_max": 1.0},   # Tabel 3.2
    "pso_seeds": tuple(range(11, 21)),                              # Bab III tahap 12; R3-R4: 10 run, seed 11-20
    "seed_akhir": (44, 45, 46, 47, 48),                             # Bab III tahap 13; R3, R6
    "R": 10_000, "seed_bootstrap": 42,                              # Bab III tahap 15; pers. (2.43)-(2.45)
    "alpha": 0.05, "m_uji": 3,                                      # pers. (2.48)-(2.49): alpha' = 0,05/3
}

# 3) Angka target (REVISI_30OKT Bagian 3; contoh Bab II tetap; nomor tahap mengikuti Bab III)
TARGET = {
    "1.1": {"baris": 9672, "kolom": ["nama", "rating", "ulasan", "tanggal", "likes", "balasan_dev", "versi"],
            "tanggal": ("2026-07-19", "2026-10-04"), "versi": ["1.6.0"],
            "per_bulan": {"2026-07": 1734, "2026-08": 3857, "2026-09": 3525, "2026-10": 556}},
    "1.2": {"kosong": {"balasan_dev": 4474}, "dibalas_pct": {"1-4": 99.6, "5": 30.8}, "likes_nol_pct": 95.1,
            "spearman": -0.2404, "r3_panjang": (315, 479), "median_kata": [15, 15, 11, 2, 2], "emoji_simbol": 549,
            "huruf_ulang": (326, 3.4), "baca_ulang": 797, "duplikat_pct": 36.7, "label_ganda": (42, 2721),
            "nama_unik": 9573},
    "2": {"baris": 9672, "kolom": ["review_id", "ulasan", "rating", "tanggal", "versi"]},
    "3": {"per_rating": {1: 1713, 2: 372, 3: 479, 4: 659, 5: 6449}, "positif": (7108, 73.49),
          "non_positif": (2564, 26.51)},
    "4": {"contoh_bab2": ["baguss bgt! tp kadang2 error, nunggu lamaa", "mantap"],
          "berubah": [66, 549, 2001, 326, 797, 547], "kosong": 92, "kosong_rating": {5: 85, 4: 6, 1: 1}, "hasil": 9580},
    "5": {"kunci": 5684, "kunci_seri": 11, "ulasan_seri": 24, "dokumen": 5673, "dibuang": (3907, 3784, 99, 24),
          "positif_pct": (73.2, 57.0), "gabungan": (59, 0)},        # gabungan: kunci terdampak, memuat angka (R9)
    "6.1": {"grup": 5567, "latih": (3967, 1708, 2259), "validasi": (853, 366, 487), "uji": (853, 367, 486)},
    "6.2": {"bocor": 0, "non_positif_pct": (43.1, 42.9, 43.0), "p_label": 0.9969,
            "p_panjang": 0.9989, "panjang_pct": {"latih": (13.7, 47.5, 38.8), "validasi": (13.6, 47.7, 38.7),
                                                 "uji": (13.4, 47.9, 38.7)}},
    "7": {"L_max": 128, "terpotong": {"latih": 0, "validasi": None, "uji": None},
          "terpotong_id": []},                                      # validasi/uji diisi setelah output 7 disetujui
    "8.1": {"T": 992, "T_w": 99},                                   # n = 3.967, rho = 0,1
    "8.3": {"t0": 0.6167, "t1": 0.6667, "C": [[2, 1], [1, 2]], "kandidat": [3, 6]},    # contoh Tabel 2.9
    "9.2": {"kappa_contoh": 0.70, "toleransi": 1e-12},              # contoh Bab II 2.3.5
    "10": {"T": 992, "T_w": 99}, "11": {"run": 10, "kandidat": 50, "pelatihan": 100},   # 11: kandidat/pelatihan per run
    "12.1": {"iter1": (5.6237, 3.6355), "iter2": (2.6718, 3.3698), "gbest_iter2": 9.8923,
             "toleransi": 5e-4, "run": 10, "evaluasi": 50},         # contoh Tabel 2.17; evaluasi per run
    "13": {"pelatihan": 15, "toleransi": 1e-4}, "14.1": {"n_uji": 853},
    "15": {"f1_contoh": 0.6703, "persentil": (0.8333, 99.1667), "toleransi": 1e-12},   # contoh Tabel 2.19
}

# 4) Uji singkat fungsi bantu, lalu ringkasan
u = lib.bantu.uji_bantu(CFG["seed_fitness"][0])
rentang = lambda s: f"{s[0]}–{s[-1]}" if list(s) == list(range(s[0], s[-1] + 1)) else ", ".join(map(str, s))
judul("0.2", "Konfigurasi dan fungsi bantu", masukan="lib/bantu.py", keluaran="PATH, CFG, TARGET (di memori)")
print(f"CFG {len(CFG)} entri · TARGET {len(TARGET)} subtahap · PATH {len(PATH)} lokasi · "
      f"fungsi bantu {len(lib.bantu.__all__)}")
print(f"seed: fitness {rentang(CFG['seed_fitness'])} · RS {rentang(CFG['rs_seeds'])} · PSO {rentang(CFG['pso_seeds'])} · "
      f"model akhir {rentang(CFG['seed_akhir'])} · GPU {CFG['gpu']}")
cek("fmt_id(7204) · fmt_id(0.836)", [fmt_id(7204), fmt_id(0.836)], ["7.204", "0,8360"])
cek("atur_seed() dua kali, angka acak sama", "ya" if u["ulang_sama"] else "tidak", "ya")
cek("mode deterministik PyTorch (KT-2)", "aktif" if u["deterministik"] else "mati", "aktif")
cek("TF32 matmul · cuDNN (KT-3)", ["nyala" if x else "mati" for x in u["tf32"]], ["mati", "mati"])
S3 = [set(CFG[k]) for k in ("seed_fitness", "rs_seeds", "pso_seeds", "seed_akhir")]
cek("run RS · PSO · seed akhir (R3) · tidak bertumpuk · kunci lama terhapus",
    [len(CFG["rs_seeds"]), len(CFG["pso_seeds"]), len(CFG["seed_akhir"]),
     "ya" if sum(map(len, S3)) == len(set().union(*S3)) else "tidak",
     "ya" if not ({"rs_seed", "kembaran_dekat"} & set(CFG) or "seed" in CFG["pso"]) else "tidak"],
    [TARGET["11"]["run"], TARGET["12.1"]["run"], TARGET["13"]["pelatihan"] // 3, "ya", "ya"])   # 15 pelatihan / 3 skema
selesai()

## Tahap 1.1 — Muat data mentah

**Dasar naskah:** Bab III tahap 1 (Pengumpulan Data) dan subbab 3.1: `review_wondr_bni_databaruu.csv`, 9.672 ulasan, 19 Juli–4 Oktober 2026, versi aplikasi 1.6.0, disimpan apa adanya dan tidak diubah (`REVISI_30OKT.md` R1). `review_id` dibuat di Tahap 2 sesuai Bab III tahap 2 (`CLAUDE.md` Bagian 10, keputusan no. 7).

**Package:** `pandas` (baca .csv, olah tabel), `pyarrow` (mesin simpan .parquet); `lib.bantu` (fungsi bantu Tahap 0.2).

In [ ]:
# ══ Tahap 1.1 — Muat data mentah ══
# Package: pandas (baca .csv, olah tabel), pyarrow (mesin simpan .parquet)
import pandas as pd
mulai()
T = TARGET["1.1"]
BULAN = ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli",
         "Agustus", "September", "Oktober", "November", "Desember"]
tgl_id = lambda t: f"{t.day} {BULAN[t.month - 1]} {t.year}"       # 2026-07-19 -> "19 Juli 2026"

# 1) Baca data mentah (.csv, R1); versi dibaca sebagai teks agar "1.6.0" tidak pernah ditafsirkan sebagai angka
mentah = pd.read_csv(PATH["csv"], dtype={"versi": str})

# 2) Buang kolom indeks bawaan (Unnamed*), bila ada; isi tidak diubah, review_id dibuat di Tahap 2
buang = [c for c in mentah.columns if str(c).startswith("Unnamed")]
df = mentah.drop(columns=buang)

# 3) Simpan, lalu pastikan berkas yang dibaca ulang identik
keluar = PATH["hasil"] / "tahap01_mentah.parquet"
df.to_parquet(keluar, index=False)
identik = df.equals(pd.read_parquet(keluar))

# 4) Ringkasan + cek
tgl = pd.to_datetime(df["tanggal"])                                # hanya untuk ringkasan; kolom asli tetap
per_bulan = {k: int(v) for k, v in tgl.dt.strftime("%Y-%m").value_counts().sort_index().items()}
versi = sorted(df["versi"].astype(str).unique())
judul("1.1", "Muat data mentah", masukan=f"{rel(PATH['csv'])} ({fmt_id(len(mentah))} baris, {mentah.shape[1]} kolom)",
      keluaran=f"{rel(keluar)} ({fmt_id(len(df))} baris, {df.shape[1]} kolom)")
print(f"Kolom dibuang  : {', '.join(map(str, buang)) or '-'}")
print(f"Rentang tanggal: {tgl_id(tgl.min())} – {tgl_id(tgl.max())}")
print(f"Versi aplikasi : {', '.join(versi)}\n\n{'Bulan':<16}{'Ulasan':>7}")
for k, v in per_bulan.items():
    print(f"{BULAN[int(k[5:]) - 1] + ' ' + k[:4]:<16}{fmt_id(v):>7}")
print(f"{'Total':<16}{fmt_id(sum(per_bulan.values())):>7}")
cek("jumlah baris", len(df), T["baris"])
cek("nama dan urutan kolom", "sesuai" if list(df.columns) == T["kolom"] else ", ".join(map(str, df.columns)), "sesuai")
cek("rentang tanggal", (str(tgl.min().date()), str(tgl.max().date())), T["tanggal"])
cek("nilai unik versi", versi, T["versi"])
cek("ulasan per bulan", per_bulan, T["per_bulan"])
cek("parquet identik dengan isi .csv", "ya" if identik else "tidak", "ya")
selesai()